In [ ]:
from pathlib import Path
import pandas as pd
from utils.data_cleaning import load_raw_data

df = load_raw_data("data")[['title', 'maintext']]
df = df.head(5000) # Ensure 5,000 rows limit

from datasets import Dataset

# Format data to include the prompt context (aligned with our generation task)
def format_prompt(row):
    return f"Article: {row['maintext']} \n\nTask: Generate a market-aligned title. \n\nTitle: {row['title']}"

df["text"] = df.apply(format_prompt, axis=1)
dataset = Dataset.from_pandas(df[['text']])

dataset_splits = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = dataset_splits["train"]
eval_dataset = dataset_splits["test"]
dataset_splits.save_to_disk("cache/financial_dataset")

print("Dataset successfully grouped, split, and saved to 'cache/financial_dataset'")
train_dataset

In [2]:
import os
from huggingface_hub import login

# login()

In [3]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024 # Choose any!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# Unsloth supports Llama, Mistral, Phi, Qwen, Gemma, and more natively.
model_id = "HuggingFaceTB/SmolLM3-3B" # Or appropriate Unsloth-supported equivalent

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "HuggingFaceTB/SmolLM3-3B", 
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
    # Add this line to skip the 120s timeout check
    disable_log_stats = True, 
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)
model.print_trainable_parameters()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Smollm3 patching. Transformers: 5.5.0.
   \\   /|    AMD Radeon RX 6800 XT. Num GPUs = 1. Max memory: 15.984 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+rocm7.2. ROCm Toolkit: 7.2.26015. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

/home/andrea/ai_work/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


HuggingFaceTB/SmolLM3-3B does not have a padding token! Will use pad_token = <|finetune_right_pad_id|>.
trainable params: 30,228,480 || all params: 3,105,327,104 || trainable%: 0.9734


In [4]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

output_dir = "./financial_llm_adapters_unsloth"

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    logging_steps=10,
    max_steps=100,
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="steps",
    save_steps=20,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    gradient_checkpointing=True,
    warmup_ratio=0.03,
    optim="adamw_8bit",
    torch_empty_cache_steps=10,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=training_args,
)

# Start SFT training
trainer_stats = trainer.train()

# Save final adapters and tokenizer
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"LoRA adapters successfully saved to: {output_dir}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=10):   0%|          | 0/4750 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=10):   0%|          | 0/250 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,750 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 30,228,480 of 3,105,327,104 (0.97% trained)


Unsloth: Will smartly offload gradients to save VRAM!


/home/andrea/ai_work/.venv/lib/python3.12/site-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/andrea/ai_work/.venv/lib/python3.12/site-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/andrea/ai_work/.venv/lib/python3.12/site-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/andrea/ai_work/.venv/lib/python3.12/site-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check

Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
import evaluate
from tqdm import tqdm

# Put Unsloth model in Fast Inference mode (2x faster generation)
FastLanguageModel.for_inference(model)

def generate_title(text_prompt, max_new_tokens=32):
    inputs = tokenizer([text_prompt], return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id,
        use_cache=True,
        do_sample=False
    )
    
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    title_segment = decoded.split("Title: ")[-1].strip()
    return title_segment

rouge = evaluate.load("rouge")
predictions = []
references = []

sample_eval = eval_dataset.select(range(min(50, len(eval_dataset))))

print("Starting evaluation loop...")
for example in tqdm(sample_eval):
    full_text = example['text']
    prompt_cut = full_text.split("Title: ")[0] + "Title: "
    reference = full_text.split("Title: ")[-1].strip()
    
    prediction = generate_title(prompt_cut)
    predictions.append(prediction)
    references.append(reference)

results = rouge.compute(predictions=predictions, references=references)
print("Evaluation Results:", results)